# Unit 1: CNN基础理论与PyTorch入门

## 学习目标
- 理解卷积神经网络的基本概念与工作原理
- 掌握PyTorch张量的基本操作
- 了解CNN核心组件：卷积、池化、激活函数
- 学会使用PyTorch进行基础张量运算

## 参考资源
- [PyTorch官方文档 - Tensors](https://pytorch.org/docs/stable/tensors.html)
- [PyTorch教程 - What is PyTorch?](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html)
- [CNN可视化解释](https://poloclub.github.io/cnn-explainer/)

## 1.1 卷积神经网络概述

卷积神经网络(Convolutional Neural Network, CNN)是一种专门用于处理具有网格结构数据(如图像)的深度学习模型。

### CNN的核心思想
1. **局部连接**：每个神经元只与输入数据的局部区域连接
2. **权值共享**：同一个卷积核在输入数据上滑动，共享相同的权重
3. **空间下采样**：通过池化操作逐步减小特征图的空间尺寸

### CNN的典型结构
```
输入图像 → 卷积层 → 激活函数 → 池化层 → 卷积层 → 激活函数 → 池化层 → 全连接层 → 输出
```

## 1.2 PyTorch环境检查

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA是否可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"GPU设备: {torch.cuda.get_device_name(0)}")

## 1.3 PyTorch张量基础

张量(Tensor)是PyTorch中最基本的数据结构，类似于NumPy的ndarray，但支持GPU加速和自动求导。

In [ ]:
print("=" * 60)
print("1.3.1 张量的创建")
print("=" * 60)

scalar_tensor = torch.tensor(5.0)
print(f"标量张量: {scalar_tensor}, 形状: {scalar_tensor.shape}")

vector_tensor = torch.tensor([1.0, 2.0, 3.0])
print(f"向量张量: {vector_tensor}, 形状: {vector_tensor.shape}")

matrix_tensor = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(f"矩阵张量:\n{matrix_tensor}, 形状: {matrix_tensor.shape}")

random_tensor = torch.randn(3, 4)
print(f"随机张量(标准正态分布):\n{random_tensor}")

zeros_tensor = torch.zeros(2, 3)
print(f"全零张量:\n{zeros_tensor}")

ones_tensor = torch.ones(2, 3)
print(f"全一张量:\n{ones_tensor}")

In [ ]:
print("=" * 60)
print("1.3.2 张量的基本操作")
print("=" * 60)

x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print(f"x:\n{x}")
print(f"y:\n{y}")
print(f"x + y:\n{x + y}")
print(f"x * y (逐元素相乘):\n{x * y}")
print(f"x @ y (矩阵乘法):\n{x @ y}")
print(f"x.mean(): {x.mean()}")
print(f"x.sum(): {x.sum()}")
print(f"x.max(): {x.max()}")

In [ ]:
print("=" * 60)
print("1.3.3 张量的形状变换")
print("=" * 60)

x = torch.arange(12)
print(f"原始张量: {x}, 形状: {x.shape}")

x_reshaped = x.reshape(3, 4)
print(f"重塑后:\n{x_reshaped}, 形状: {x_reshaped.shape}")

x_transposed = x_reshaped.t()
print(f"转置后:\n{x_transposed}, 形状: {x_transposed.shape}")

x_squeezed = torch.zeros(1, 3, 1, 4)
print(f"压缩前形状: {x_squeezed.shape}")
print(f"压缩后形状: {x_squeezed.squeeze().shape}")

## 1.4 卷积运算原理

卷积是CNN的核心操作。卷积核(kernel/filter)在输入数据上滑动，计算局部区域的加权和。

### 关键参数
- **卷积核大小(kernel_size)**：卷积核的空间尺寸
- **步长(stride)**：卷积核每次滑动的步长
- **填充(padding)**：在输入边缘添加的零值数量

### 输出尺寸计算公式
```
output_size = (input_size + 2 * padding - kernel_size) / stride + 1
```

In [ ]:
print("=" * 60)
print("1.4.1 手动实现2D卷积")
print("=" * 60)

def conv2d_manual(input_matrix, kernel):
    """
    手动实现2D卷积操作
    
    参数:
        input_matrix: 输入矩阵 (H, W)
        kernel: 卷积核 (kH, kW)
    返回:
        output: 卷积结果
    """
    input_h, input_w = input_matrix.shape
    kernel_h, kernel_w = kernel.shape
    
    output_h = input_h - kernel_h + 1
    output_w = input_w - kernel_w + 1
    
    output = torch.zeros(output_h, output_w)
    
    for i in range(output_h):
        for j in range(output_w):
            output[i, j] = (input_matrix[i:i+kernel_h, j:j+kernel_w] * kernel).sum()
    
    return output

input_img = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0, 12.0],
    [13.0, 14.0, 15.0, 16.0]
])

kernel = torch.tensor([
    [1.0, 0.0],
    [0.0, -1.0]
])

conv_result = conv2d_manual(input_img, kernel)
print(f"输入矩阵:\n{input_img}")
print(f"卷积核:\n{kernel}")
print(f"卷积结果:\n{conv_result}")

In [ ]:
print("=" * 60)
print("1.4.2 使用PyTorch实现2D卷积")
print("=" * 60)

import torch.nn.functional as F

input_tensor = input_img.unsqueeze(0).unsqueeze(0)
kernel_tensor = kernel.unsqueeze(0).unsqueeze(0)

print(f"输入张量形状: {input_tensor.shape}")
print(f"卷积核形状: {kernel_tensor.shape}")

conv_output = F.conv2d(input_tensor, kernel_tensor)
print(f"PyTorch卷积结果:\n{conv_output.squeeze()}")

## 1.5 池化操作

池化(Pooling)用于降低特征图的空间尺寸，减少参数数量，提高模型的平移不变性。

### 常见池化类型
- **最大池化(Max Pooling)**：取窗口内的最大值
- **平均池化(Average Pooling)**：取窗口内的平均值

In [ ]:
print("=" * 60)
print("1.5 池化操作演示")
print("=" * 60)

x = torch.tensor([
    [1.0, 3.0, 2.0, 4.0],
    [5.0, 6.0, 1.0, 2.0],
    [3.0, 2.0, 7.0, 8.0],
    [9.0, 1.0, 2.0, 4.0]
])

x_input = x.unsqueeze(0).unsqueeze(0)

max_pool_output = F.max_pool2d(x_input, kernel_size=2, stride=2)
print(f"输入:\n{x}")
print(f"最大池化结果(2x2):\n{max_pool_output.squeeze()}")

avg_pool_output = F.avg_pool2d(x_input, kernel_size=2, stride=2)
print(f"平均池化结果(2x2):\n{avg_pool_output.squeeze()}")

## 1.6 激活函数

激活函数为神经网络引入非线性，使模型能够学习复杂的模式。

### 常用激活函数
- **ReLU**: f(x) = max(0, x)
- **Sigmoid**: f(x) = 1 / (1 + e^(-x))
- **Tanh**: f(x) = (e^x - e^(-x)) / (e^x + e^(-x))

In [ ]:
print("=" * 60)
print("1.6 激活函数可视化")
print("=" * 60)

x = torch.linspace(-5, 5, 100)

relu_out = torch.relu(x)
sigmoid_out = torch.sigmoid(x)
tanh_out = torch.tanh(x)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(x.numpy(), relu_out.numpy(), 'b-', linewidth=2)
axes[0].set_title('ReLU', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].axvline(x=0, color='k', linestyle='-', linewidth=0.5)

axes[1].plot(x.numpy(), sigmoid_out.numpy(), 'r-', linewidth=2)
axes[1].set_title('Sigmoid', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].axvline(x=0, color='k', linestyle='-', linewidth=0.5)

axes[2].plot(x.numpy(), tanh_out.numpy(), 'g-', linewidth=2)
axes[2].set_title('Tanh', fontsize=14)
axes[2].grid(True, alpha=0.3)
axes[2].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[2].axvline(x=0, color='k', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 1.7 设备管理(CPU/GPU)

PyTorch支持将张量和模型移动到GPU上进行加速计算。

In [ ]:
print("=" * 60)
print("1.7 设备管理")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前设备: {device}")

x_cpu = torch.randn(1000, 1000)
x_gpu = x_cpu.to(device)
print(f"张量所在设备: {x_gpu.device}")

import time

start = time.time()
for _ in range(100):
    result_cpu = x_cpu @ x_cpu
cpu_time = time.time() - start
print(f"CPU计算时间: {cpu_time:.4f}秒")

if torch.cuda.is_available():
    start = time.time()
    for _ in range(100):
        result_gpu = x_gpu @ x_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"GPU计算时间: {gpu_time:.4f}秒")
    print(f"加速比: {cpu_time/gpu_time:.2f}x")

## 1.8 综合练习：构建简单的CNN前向传播

In [ ]:
print("=" * 60)
print("1.8 简单CNN前向传播演示")
print("=" * 60)

import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=4, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    
    def forward(self, x):
        print(f"输入形状: {x.shape}")
        x = self.conv1(x)
        print(f"卷积后形状: {x.shape}")
        x = self.relu(x)
        x = self.pool(x)
        print(f"池化后形状: {x.shape}")
        return x

model = SimpleCNN()
print(f"模型结构:\n{model}")

dummy_input = torch.randn(1, 1, 8, 8)
output = model(dummy_input)
print(f"\n最终输出形状: {output.shape}")

## 本章小结

本单元我们学习了：
1. CNN的基本概念和工作原理
2. PyTorch张量的创建与基本操作
3. 卷积、池化、激活函数的原理与实现
4. CPU/GPU设备管理
5. 简单CNN的前向传播过程

## 练习建议
1. 尝试修改卷积核的大小和参数，观察输出变化
2. 比较不同激活函数的特点
3. 练习张量的各种形状变换操作
4. 尝试构建不同结构的简单CNN

## 下一步
进入Unit 2，深入学习PyTorch的自动求导机制和张量高级操作。